In [1]:
import re

# ====================== 【仅需修改这2个参数】 ======================
INPUT_PATH = "骆驼祥子.txt"    # 你的小说文件路径
BLOCK_WORDS = 1200            # 分块字数阈值（达标后等段落结束再分割）
# ==================================================================

def clean_novel(text):
    """清洗文本：删除中文章节+等号，清理多余空行"""
    # 匹配 第+中文数字+章 + 换行 + 连续等号 → 替换为换行
    chapter_pattern = r"第[零一二三四五六七八九十百]+章\n=+\n"
    text = re.sub(chapter_pattern, "\n", text)
    # 合并多余空行，去除每行空白，过滤空行
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    return "\n".join(lines)

def split_to_blocks(clean_text, max_words):
    """
    核心分块函数：返回 list[str]
    达到字数阈值 → 等待第一个段落结束再分块
    """
    blocks = []
    current_block = []
    total_words = 0

    # 按自然段遍历
    for paragraph in clean_text.split("\n"):
        para_length = len(paragraph)
        # 超过阈值且当前块有内容 → 保存块，新建块
        if total_words + para_length > max_words and current_block:
            blocks.append("\n".join(current_block))
            current_block = [paragraph]
            total_words = para_length
        else:
            current_block.append(paragraph)
            total_words += para_length

    # 加入最后一个块
    if current_block:
        blocks.append("\n".join(current_block))
    
    # 返回值就是：list[str]
    return blocks    

In [2]:
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

cleaned_text = clean_novel(raw_text)

In [3]:
train_data_list = split_to_blocks(cleaned_text, BLOCK_WORDS)

In [4]:
print(f"列表长度：{len(train_data_list)} 个文本块")

列表长度：126 个文本块


In [5]:
with open("训练数据_list.txt", "w", encoding="utf-8") as f:
    f.write(str(train_data_list))

In [6]:
from datasets import Dataset

In [7]:
myDataTest = Dataset.from_dict({"text": train_data_list})

In [8]:
myDataTest

Dataset({
    features: ['text'],
    num_rows: 126
})

In [9]:
# 保存Dataset到本地文件夹（自动生成Arrow格式数据，完全保留结构）
myDataTest.save_to_disk("./my_novel_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/126 [00:00<?, ? examples/s]

In [10]:
def get_training_corpus(data):
    d = data['text']
    batch_size = 10
    for i in range(0, len(d), batch_size):
        samples = d[i: i + batch_size]
        yield samples

In [11]:
from transformers import AutoTokenizer

In [13]:
tokenizer_gpt2 = AutoTokenizer.from_pretrained('gpt2')

ConnectTimeout: [WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。

In [ ]:
# 词汇表大小：3000足够
my_tokenizer = tokenizer_gpt2.train_new_from_iterator(
    get_training_corpus(myDataTest),
    vocab_size = 8000,
    min_frequency = 2,
)

In [ ]:
test_code = '''他不怕吃苦，也没有一般洋车夫的可以原谅而不便效法的恶习，他的聪明和努力都足以使他的志愿成为事实。假若他的环境好一些，或多受着点教育，他一定不会落在"胶皮团"⑤里，而且无论是干什么，他总不会辜负了他的机会。不幸，他必须拉洋车；好，在这个营生里他也证明出他的能力与聪明。他仿佛就是在地狱里也能作个好鬼似的。生长在乡间，失去了父母与几亩薄田，十八岁的时候便跑到城里来。带着乡间小伙子的足壮与诚实，凡是以卖力气就能吃饭的事他几乎全作过了。可是，不久他就看出来，拉车是件更容易挣钱的事；作别的苦工，收入是有限的；拉车多着一些变化与机会，不知道在什么时候与地点就会遇到一些多于所希望的报酬。自然，他也晓得这样的机遇不完全出于偶然，而必须人与车都得漂亮精神，有货可卖才能遇到识货的人。想了一想，他相信自己有那个资格：他有力气，年纪正轻；所差的是他还没有跑过，与不敢一上手就拉漂亮的车。但这不是不能胜过的困难，有他的身体与力气作基础，他只要试验个十天半月的，就一定能跑得有个样子，然后去赁辆新车，说不定很快的就能拉上包车，然后省吃俭用的一年二年，即使是三四年，他必能自己打上一辆车，顶漂亮的车！看着自己的青年的肌肉，他以为这只是时间的问题，这是必能达到的一个志愿与目的，绝不是梦想！'''

'|'.join([my_tokenizer.decode(i) for i in my_tokenizer.encode(test_code)])

In [ ]:
my_tokenizer.save_pretrained("./my_novel_tokenizer")